# 05 — Responses vs Agent-Owned State

## Goal

Understand where conversational history and execution state should live
in a Foundry-hosted Deep Agent.

We will compare two designs:

1. Foundry Responses owns conversation history
2. LangGraph owns conversation + execution state

For this project, we choose:

> **Responses = client-facing protocol**
>
> **LangGraph = persistent agent-state authority**

We are making the architectural decision in this notebook.
Checkpoint persistence itself will be implemented later.

## Three independent architecture layers

### 1. Protocol

How clients communicate with the agent.

For this project:

`Responses`

### 2. Agent runtime

Where reasoning, tools, planning, and orchestration happen.

For this project:

`Deep Agents + LangGraph`

### 3. Persistence

Where durable agent state lives.

Planned direction:

- SQLite during development
- Postgres for production

```
Client
   │
   │ Responses protocol
   ▼
Foundry Hosted Agent
   │
   ▼
Deep Agents / LangGraph
   │
   ▼
Persistent state
SQLite → Postgres
```

## Responses is the northbound protocol, not necessarily the model protocol

A Hosted Agent may expose the Responses protocol even if the model inside
the agent is not an OpenAI model.

```
                         northbound

Client
   │
   │ Responses
   ▼
Hosted Agent
   │
   ▼
LangGraph
   │
   ├──────── GPT
   │
   ├──────── Claude
   │
   └──────── another model
                         southbound
```

## Option A — Foundry owns conversation history

Foundry Responses stores the transcript and supplies prior history
to the Hosted Agent when a conversation continues.

```
Client
   ↓
Responses conversation
   ↓
Foundry history store
   ↓
handler
   ↓
Deep Agent
```

Advantages:
+ very little application persistence code
+ platform-managed conversation continuity
+ convenient for straightforward chat agents

Trade-offs:
- conversation state and graph state live in different systems
- agent persistence becomes coupled to the hosting platform
- more care is needed if LangGraph later also persists messages


## Option B — LangGraph owns conversation + execution state

Responses remains the external API contract, but LangGraph becomes the
source of truth for persistent agent state.

```
Client
   ↓
Responses request
   ↓
Hosted Agent handler
   ↓
thread_id
   ↓
LangGraph checkpointer
   ├── messages
   ├── graph state
   ├── interrupts
   └── checkpoints
   ↓
Deep Agent
```




## A checkpointer is not just chat-history storage

LangGraph checkpoints the graph state.

If `messages` is part of that state, conversation history is persisted
along with the rest of the agent's state.

thread_id = research-123

```
checkpoint 1
├── messages
│   └── User: Research company X
├── todos
└── other graph state


checkpoint 2
├── messages
│   ├── User: Research company X
│   └── Assistant: ...
├── todos
└── other graph state
```

In [1]:
from deep_agents_foundry import build_research_agent

agent = build_research_agent()

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


In [2]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain Hosted Agents in one sentence.",
            }
        ]
    }
)

print(result.keys())

dict_keys(['messages', 'files'])


In [3]:
for key, value in result.items():
    print(f"\n### {key}")
    print(type(value))


### messages
<class 'list'>

### files
<class 'dict'>


In [4]:
messages = result["messages"]

print("Number of messages:", len(messages))

for i, message in enumerate(messages):
    print(i, type(message).__name__)

Number of messages: 2
0 HumanMessage
1 AIMessage


The conversation transcript is already part of LangGraph state.

A future checkpointer can therefore persist it together with other
graph state rather than requiring a separate chat-history mechanism.

Later, the checkpointer will use `thread_id` as the durable identity of an
agent thread.

The intended model is:

```
thread_id
   ↓
checkpoint store
   ↓
current LangGraph state
```

## State ownership rule

For this project:

### Responses owns

- client-facing API contract
- request/response schema
- streaming protocol
- Hosted Agent integration

### LangGraph owns

- conversation messages
- graph state
- checkpoints
- interrupts/resume state
- durable execution state

### Therefore

We will not also hydrate Foundry-managed conversation history into the same
LangGraph thread.

That would create two sources of truth for the transcript.

## State ownership also affects token economics

If both Foundry and LangGraph replay the same historical messages, the model
may receive duplicate or unnecessarily large context.

A single state authority makes future context policies easier to implement:

- trim old messages
- summarize prior turns
- selectively retrieve context
- externalize intermediate work
- keep only useful checkpoints

# Architecture decision

For `deep-agents-on-foundry`:

## Northbound API

Microsoft Foundry Hosted Agent
using the Responses protocol.

## Agent runtime

Deep Agents + LangGraph.

## Persistent state authority

LangGraph checkpointer.

Development:
SQLite

Production:
Postgres

## Conversation history

Stored as part of LangGraph thread state rather than separately replayed from
Foundry Responses conversation history.